In [4]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import Dataset, DataLoader
from linformer import Linformer 
from itertools import chain   
from vit_pytorch.efficient import ViT   
from tqdm.notebook import tqdm   
from __future__ import print_function  

data = {}
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
#device = torch.device("cpu")
#data['test']  = np.load('drive/MyDrive/dataverse_files/Dataset5_raw_test.npz')
data['train'] = dict(np.load('dataverse_files/Dataset5_raw_train.npz'))
data['val']   = dict(np.load('dataverse_files/Dataset5_raw_val.npz'))


In [2]:
!pip install linformer
!pip install vit_pytorch

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.3/87.3 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 5.9 MB/s eta 0:00:00


In [5]:
mean = data['train']['image'].mean(axis=(0,1,2), keepdims=True)
devia = data['train']['image'].std(axis=(0,1,2), keepdims=True)
data['train']['image'] = (data['train']['image'] - mean) / devia
data['val']['image'] = (data['val']['image'] - mean) / devia

In [6]:
class CustomDataset(Dataset):
    def __init__(self, data, labels, transform):
        self.transform = transform
        self.data = data
        self.labels = labels 
    
    def __len__(self):
        return self.labels.size

    def __getitem__(self, idx):
        temp = int(self.labels[idx])
        return self.transform(self.data[idx]).float(), torch.tensor(temp)

In [7]:
transform = transforms.Compose([transforms.ToTensor()])
print(data['train']['image'].transpose(0, 3, 1, 2).shape)
trainset = CustomDataset(data['train']['image'], data['train']['image_label'], transform)
train_loader = DataLoader(trainset, 64, shuffle=True)

valset = CustomDataset(data['val']['image'], data['val']['image_label'], transform)
val_loader = DataLoader(valset, 1, shuffle=True)

(19610, 3, 224, 224)


In [9]:
efficient_transformer = Linformer(
    dim=128,
    seq_len=49+1,  # 7x7 patches + 1 cls-token
    depth=12,
    heads=8,
    k=64
)

model = ViT(
    dim=128,
    image_size=224,
    patch_size=32,
    num_classes=100,
    transformer=efficient_transformer,
    channels=3
).to(device)

class Model(nn.Module):
  def __init__(self):
    super().__init__()

    self.fcn = nn.Sequential(
            nn.Linear(100, 50),
            nn.ReLU(),
            nn.Linear(50, 5),
            nn.Softmax(),
        )
    
  def forward(self, inp):
    return self.fcn(inp)

fcn = Model().to(device)
model.load_state_dict(torch.load("vit_fext_85.pth"))
model.eval()

ViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=32, p2=32)
    (1): LayerNorm((3072,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=3072, out_features=128, bias=True)
    (3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Linformer(
    (net): SequentialSequence(
      (layers): ModuleList(
        (0-11): 12 x ModuleList(
          (0): PreNorm(
            (fn): LinformerSelfAttention(
              (to_q): Linear(in_features=128, out_features=128, bias=False)
              (to_k): Linear(in_features=128, out_features=128, bias=False)
              (to_v): Linear(in_features=128, out_features=128, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
              (to_out): Linear(in_features=128, out_features=128, bias=True)
            )
            (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          )
          (1): PreNorm(
            (fn): Fe

In [10]:
import shutil
i = 9
file = f'vit_fext_{i}.pth'
shutil.copyfile(file, "drive/MyDrive/"+file)

FileNotFoundError: ignored

In [ ]:
import shutil

EPOCHS = 110
lossfn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=0)
max_correct = 0
val_count = data['val']['image'].shape[0]

train_loss = []
val_loss   = []
val_correct = []
file = "" 
train_batches = data['train']['image'].shape[0] / 64
val_batches   = data['train']['image'].shape[0]

for i in range(EPOCHS):
  lss = 0
  for x, y in train_loader:
    optimizer.zero_grad()

    features = model(x.to(device))
    y_pred = fcn.forward(features)
    loss = lossfn(y_pred, y.to(device))

    loss.backward()
    optimizer.step()
    lss += loss

  train_loss.append(lss)
  lss = 0
  correct = 0

  with torch.no_grad():
    model.eval()

    for x, y in val_loader:
      features = model(x.to(device))
      y_pred = fcn.forward(features)
      loss = lossfn(y_pred, y.to(device))

      correct += (y_pred.argmax(axis=1) == y.to(device)).sum()
      lss += loss

    val_correct.append(100*correct/val_count)
    val_loss.append(lss)
    if max_correct < correct:
      max_correct = correct
      torch.save(model.state_dict(), f'vit_fext_{i}.pth')
      file = f'vit_fext_{i}.pth'

  val_loss[-1] = val_loss[-1] / val_batches
  train_loss[-1] = train_loss[-1] / train_batches

  print("EPOCH -> ", i)
  print("val error      = ", val_loss[-1])
  print("train error    = ", train_loss[-1])
  print("val correct %  = ", val_correct[-1])

shutil.copyfile(file, "drive/MyDrive/"+file)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/container.py:217: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  input = module(input)


EPOCH ->  0
val error      =  tensor(0.4252, device='cuda:0')
train error    =  tensor(1.2649, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(66.4524, device='cuda:0')
EPOCH ->  1
val error      =  tensor(0.4218, device='cuda:0')
train error    =  tensor(1.2549, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(66.5748, device='cuda:0')
EPOCH ->  2
val error      =  tensor(0.4180, device='cuda:0')
train error    =  tensor(1.2429, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(67.3707, device='cuda:0')
EPOCH ->  3
val error      =  tensor(0.4219, device='cuda:0')
train error    =  tensor(1.2334, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(65.1209, device='cuda:0')
EPOCH ->  4
val error      =  tensor(0.4150, device='cuda:0')
train error    =  tensor(1.2236, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(68.0594, device='cuda:0')
EPOCH ->  5
val error      =  tensor(0.4125, device='cuda:0')
tra

'drive/MyDrive/vit_fext_85.pth'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [11]:
import shutil

EPOCHS = 100
lossfn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.00001, weight_decay=0)
max_correct = 0
val_count = data['val']['image'].shape[0]

train_loss = []
val_loss   = []
val_correct = []
file = "" 
train_batches = data['train']['image'].shape[0] / 64
val_batches   = data['train']['image'].shape[0]

for i in range(EPOCHS):
  lss = 0
  for x, y in train_loader:
    optimizer.zero_grad()

    features = model(x.to(device))
    y_pred = fcn.forward(features)
    loss = lossfn(y_pred, y.to(device))

    loss.backward()
    optimizer.step()
    lss += loss

  train_loss.append(lss)
  lss = 0
  correct = 0

  with torch.no_grad():
    model.eval()

    for x, y in val_loader:
      features = model(x.to(device))
      y_pred = fcn.forward(features)
      loss = lossfn(y_pred, y.to(device))

      correct += (y_pred.argmax(axis=1) == y.to(device)).sum()
      lss += loss

    val_correct.append(100*correct/val_count)
    val_loss.append(lss)
    if max_correct < correct:
      max_correct = correct
      torch.save(model.state_dict(), f'vit_fext_{i}.pth')
      file = f'vit_fext_{i}.pth'

  val_loss[-1] = val_loss[-1] / val_batches
  train_loss[-1] = train_loss[-1] / train_batches

  print("EPOCH -> ", i)
  print("val error      = ", val_loss[-1])
  print("train error    = ", train_loss[-1])
  print("val correct %  = ", val_correct[-1])

shutil.copyfile(file, "drive/MyDrive/"+file)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/container.py:217: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  input = module(input)


EPOCH ->  0
val error      =  tensor(0.4334, device='cuda:0')
train error    =  tensor(1.3813, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(70.5846, device='cuda:0')
EPOCH ->  1
val error      =  tensor(0.4170, device='cuda:0')
train error    =  tensor(1.2089, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(71.0285, device='cuda:0')
EPOCH ->  2
val error      =  tensor(0.4074, device='cuda:0')
train error    =  tensor(1.1554, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(74.0894, device='cuda:0')
EPOCH ->  3
val error      =  tensor(0.4014, device='cuda:0')
train error    =  tensor(1.1156, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(76.5381, device='cuda:0')
EPOCH ->  4
val error      =  tensor(0.3976, device='cuda:0')
train error    =  tensor(1.0846, device='cuda:0', grad_fn=<DivBackward0>)
val correct %  =  tensor(76.6758, device='cuda:0')
EPOCH ->  5
val error      =  tensor(0.3913, device='cuda:0')
tra

'drive/MyDrive/vit_fext_71.pth'